In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, lit
from pyspark.sql.types import IntegerType, DecimalType
import os
import time
import psycopg2
from pyspark.sql.utils import AnalysisException

# =============================================================================
# 1. Configuração e Inicialização
# =============================================================================

print("Iniciando a sessão Spark...")
spark = SparkSession.builder \
    .appName("Formula1_ETL_Silver_to_Gold_v1") \
    .getOrCreate()
print("Sessão Spark iniciada com sucesso!")

# Configurações de Conexão com o PostgreSQL (Reutilizando variáveis do docker-compose)
jdbc_hostname = os.getenv("DB_HOST", "db")
jdbc_port     = os.getenv("DB_PORT", "5432")
jdbc_database = os.getenv("DB_NAME", "f1database")
db_user       = os.getenv("DB_USER", "f1user")
db_password   = os.getenv("DB_PASSWORD", "1234")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
    "user": db_user, 
    "password": db_password, 
    "driver": "org.postgresql.Driver",
}

# Esquemas e Tabelas
SILVER_TABLE = "ResultadosCorridas"
GOLD_SCHEMA = "gold"

# =============================================================================
# 2. Conexão ao Banco de Dados (Verificação)
# =============================================================================

retries = 10
wait_seconds = 5
for i in range(retries):
    try:
        print("Tentando conectar ao banco de dados...")
        conn = psycopg2.connect(host=jdbc_hostname, dbname=jdbc_database, user=db_user, password=db_password, port=jdbc_port)
        conn.close()
        print("✅ Conexão com o banco de dados bem-sucedida!")
        break
    except psycopg2.OperationalError:
        print(f"⏳ Banco de dados não está pronto. Tentando novamente em {wait_seconds} segundos...")
        time.sleep(wait_seconds)
        if i == retries - 1:
            print("❌ Não foi possível conectar ao banco de dados. Abortando.")
            spark.stop()
            exit(1)

# =============================================================================
# 3. Leitura da Camada Silver
# =============================================================================

print(f"\nLendo tabela da Camada Silver: {SILVER_TABLE}")
try:
    df_silver = spark.read.jdbc(url=jdbc_url, table=SILVER_TABLE, properties=connection_properties)
    
    # --- CAST EXPLÍCITO DE CHAVES (DEBUG TYPE-MISMATCH) ---
    print("Aplicando CAST explícito nas colunas ID (chaves de negócio)...")
    df_silver = df_silver.withColumn("id_equipe", col("id_equipe").cast(IntegerType())) \
                         .withColumn("id_piloto", col("id_piloto").cast(IntegerType())) \
                         .withColumn("id_corrida", col("id_corrida").cast(IntegerType())) \
                         .withColumn("id_status", col("id_status").cast(IntegerType()))
    # -----------------------------------------------------

    count_silver = df_silver.count()
    print(f"✅ Dados da Silver carregados. Total de registros LIDOS: {count_silver}")
    if count_silver == 0:
        print("⚠️ AVISO: A Camada Silver está vazia. O ETL GOLD também ficará vazio. Verifique o job Raw->Silver.")
        spark.stop()
        exit(0)
        
    df_silver.printSchema()
except AnalysisException as e:
    print(f"❌ Erro ao ler a tabela {SILVER_TABLE}. Verifique se a tabela existe e se as permissões estão corretas. Erro: {e}")
    spark.stop()
    exit(1)

# =============================================================================
# 4. Criação e Carga das Dimensões (GOLD)
# =============================================================================

# Define a função para salvar a Dimensão e criar um nome de tabela completo
def save_dimension(df_dim, dim_name, id_col_silver, other_cols, cols_to_drop=None):
    """
    Salva dados na dimensão. A coluna ID_COL_SILVER é renomeada para CHAVE_..._ORIGEM (Business Key)
    e as colunas de drop são removidas para evitar conflito com colunas geradas.
    """
    table_name = f"{GOLD_SCHEMA}.dm_{dim_name}"
    business_key_col = f"chave_{dim_name}_origem"
    
    # 1. Selecionar e Renomear a Chave de Negócio (ID -> CHAVE_ORIGEM)
    df_dim_unique = df_dim.select(
        col(id_col_silver).alias(business_key_col), # RENOMEIA A CHAVE DE NEGÓCIO DA SILVER PARA EVITAR "ID"
        *other_cols
    ).distinct().dropna(subset=[business_key_col])

    # 1.5. REMOVER COLUNAS CONFLITANTES / GERADAS antes da inserção
    if cols_to_drop:
        df_dim_unique = df_dim_unique.drop(*cols_to_drop)
    
    print(f"\n---> Criando e carregando Dimensão: {table_name}")
    count_unique = df_dim_unique.count()
    print(f"Número de registros únicos a serem inseridos: {count_unique}")
    
    if count_unique == 0:
        print("AVISO: DataFrame da Dimensão está vazio antes da escrita no banco!")
        return table_name

    # 2. Carga no PostgreSQL com TRY/EXCEPT e Verificação Pós-Escrita
    try:
        # AQUI O SPARK GERA O INSERT: INSERT INTO gold.dm_piloto (chave_piloto_origem, primeiro_nome, sobrenome) VALUES (...)
        df_dim_unique.write \
            .mode("append") \
            .jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
        
        # --- VERIFICAÇÃO PÓS-ESCRITA (CRÍTICA) ---
        df_check = spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
        count_check = df_check.count()
        
        if count_check >= count_unique:
            print(f"✅ Dimensão {dim_name} carregada com sucesso! Registros CONFIRMADOS no PG: {count_check}")
        else:
            print(f"⚠️ AVISO DE CONTAGEM: O número de registros confirmados ({count_check}) é menor que o esperado ({count_unique}).")

    except Exception as e:
        print(f"❌ ERRO FATAL AO GRAVAR NA DIMENSÃO {table_name}: {e}")
        # Lança o erro para parar o job
        raise e
    
    return table_name

# --- 4.1. DM_PILOTO ---
# CORREÇÃO DE ESQUEMA E REMOÇÃO DE COLUNA GERADA
df_piloto_src = df_silver.select(
    "id_piloto", 
    col("primeiro_nome_piloto").alias("primeiro_nome"),  
    col("sobrenome_piloto").alias("sobrenome")          
).withColumn("nome_completo", concat_ws(" ", col("primeiro_nome"), col("sobrenome")))

# Colunas a serem enviadas: primeiro_nome, sobrenome. nome_completo será removida antes da inserção
save_dimension(
    df_piloto_src, 
    "piloto", 
    "id_piloto", # Coluna da Silver usada como chave de negócio
    ["primeiro_nome", "sobrenome", "nome_completo"], 
    cols_to_drop=["nome_completo"] # Coluna que o PG gera e não aceita inserção
)

# --- 4.2. DM_EQUIPE ---
save_dimension(
    df_silver, 
    "equipe", 
    "id_equipe", 
    ["nome_equipe"]
)

# --- 4.3. DM_CORRIDA ---
save_dimension(
    df_silver, 
    "corrida", 
    "id_corrida", 
    ["ano", "rodada", "nome_corrida"]
)

# --- 4.4. DM_STATUS ---
save_dimension(
    df_silver, 
    "status", 
    "id_status", 
    ["descricao_status"]
)

# =============================================================================
# 5. Criação e Carga da Tabela de Fato (FT_VOLTAS_TEMPO_PARADA)
# =============================================================================

print("\n\n=== Iniciando a construção da Tabela de Fato ===")

# --- 5.1. Leitura das Dimensões para obter as SRKs ---
def read_dimension_with_srk(dim_name):
    """
    Lê a Dimensão da Gold, pegando SRK e Chave de Negócio.
    A Chave de Negócio é RENOMEADA para ID_... para o JOIN com a tabela Silver.
    """
    table_name = f"{GOLD_SCHEMA}.dm_{dim_name}"
    srk_col = f"srk_{dim_name}"
    # NOVO NOME DA COLUNA DE ORIGEM NO DDL GOLD
    chave_origem_col = f"chave_{dim_name}_origem" 
    
    df_dim = spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
    
    # Retorna o SRK (srk_piloto) e a Chave de Negócio (chave_piloto_origem)
    # A Chave de Negócio é RENOMEADA DE VOLTA PARA ID_... para casar com a coluna da tabela Silver (df_silver)
    return df_dim.select(col(srk_col), col(chave_origem_col).alias(f"id_{dim_name}"))

# Lendo SRKs
df_dim_piloto   = read_dimension_with_srk("piloto")
df_dim_equipe   = read_dimension_with_srk("equipe")
df_dim_corrida  = read_dimension_with_srk("corrida")
df_dim_status   = read_dimension_with_srk("status")

# DEBUG: Contagem para ver se as SKs voltaram
print(f"DEBUG: SRKs de Piloto lidas do Gold: {df_dim_piloto.count()}")


# --- 5.2. Juntando Silver com Dimensões para obter as SRKs ---
# Fazemos o JOIN usando as colunas ID_... da Silver e as colunas renomeadas (ID_...) no read_dimension_with_srk
df_fato_base = df_silver.alias("S") \
    .join(df_dim_piloto.alias("DP"), col("S.id_piloto") == col("DP.id_piloto"), "inner") \
    .join(df_dim_equipe.alias("DE"), col("S.id_equipe") == col("DE.id_equipe"), "inner") \
    .join(df_dim_corrida.alias("DC"), col("S.id_corrida") == col("DC.id_corrida"), "inner") \
    .join(df_dim_status.alias("DS"), col("S.id_status") == col("DS.id_status"), "inner")

# DEBUG: Contagem após todos os JOINs
count_fato_base = df_fato_base.count()
print(f"DEBUG: Registros após todos os JOINs: {count_fato_base}")
if count_fato_base == 0:
    print("❌ ERRO CRÍTICO: Zero registros após JOINs! As chaves de negócio (IDs) da Silver não bateram com as chaves de origem da Gold. Verifique o DDL Gold (UNIQUE) e a tipagem.")
    spark.stop()
    exit(1)


# --- 5.3. Seleção Final para a Tabela de Fato ---
df_fato = df_fato_base.select(
    # Seleção final para a Tabela Fato
    col("DP.srk_piloto").alias("srk_piloto"),
    col("DE.srk_equipe").alias("srk_equipe"),
    col("DC.srk_corrida").alias("srk_corrida"),
    col("DS.srk_status").alias("srk_status"),
    
    # Métricas e Atributos de Fato
    col("S.volta").cast(IntegerType()),
    col("S.posicao_na_volta").cast(IntegerType()),
    col("S.tempo_volta_ms").cast(IntegerType()),
    col("S.duracao_parada_seg").cast(DecimalType(10, 3))
)

# Remoção de nulos críticos 
df_fato = df_fato.filter(col("volta").isNotNull() & col("tempo_volta_ms").isNotNull())

print("\nEstrutura final da Tabela de Fato:")
df_fato.printSchema()
print(f"Total de registros na Fato (após filtro): {df_fato.count()}")


# --- 5.4. Carga da Tabela de Fato ---
FACT_TABLE = f"{GOLD_SCHEMA}.ft_voltas_tempo_parada"
print(f"\nIniciando carga na tabela de Fato: {FACT_TABLE}")

try:
    df_fato.write \
        .option("truncate", "true") \
        .mode("overwrite") \
        .jdbc(url=jdbc_url, table=FACT_TABLE, properties=connection_properties)
    print(f"✅ Tabela de Fato '{FACT_TABLE}' populada com sucesso!")
except Exception as e:
    print(f"❌ ERRO FATAL AO GRAVAR NA TABELA DE FATO {FACT_TABLE}: {e}")
    raise e

# =============================================================================
# 6. Finalização
# =============================================================================

print("\n🚀 Job ETL (Silver → Gold) finalizado com sucesso!")
spark.stop()

Iniciando a sessão Spark...


Sessão Spark iniciada com sucesso!
Tentando conectar ao banco de dados...
✅ Conexão com o banco de dados bem-sucedida!

Lendo tabela da Camada Silver: ResultadosCorridas


Aplicando CAST explícito nas colunas ID (chaves de negócio)...


✅ Dados da Silver carregados. Total de registros LIDOS: 319855
root
 |-- id_equipe: integer (nullable = true)
 |-- nome_equipe: string (nullable = true)
 |-- id_piloto: integer (nullable = true)
 |-- primeiro_nome_piloto: string (nullable = true)
 |-- sobrenome_piloto: string (nullable = true)
 |-- id_corrida: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- rodada: integer (nullable = true)
 |-- nome_corrida: string (nullable = true)
 |-- id_status: integer (nullable = true)
 |-- descricao_status: string (nullable = true)
 |-- volta: integer (nullable = true)
 |-- posicao_na_volta: integer (nullable = true)
 |-- tempo_volta_ms: integer (nullable = true)
 |-- duracao_parada_seg: decimal(10,3) (nullable = true)




---> Criando e carregando Dimensão: gold.dm_piloto


...

DEBUG: SRKs de Piloto lidas do Gold: 77


DEBUG: Registros após todos os JOINs: 319855



Estrutura final da Tabela de Fato:
root
 |-- srk_piloto: integer (nullable = true)
 |-- srk_equipe: integer (nullable = true)
 |-- srk_corrida: integer (nullable = true)
 |-- srk_status: integer (nullable = true)
 |-- volta: integer (nullable = true)
 |-- posicao_na_volta: integer (nullable = true)
 |-- tempo_volta_ms: integer (nullable = true)
 |-- duracao_parada_seg: decimal(10,3) (nullable = true)



Total de registros na Fato (após filtro): 319683

Iniciando carga na tabela de Fato: gold.ft_voltas_tempo_parada


✅ Tabela de Fato 'gold.ft_voltas_tempo_parada' populada com sucesso!

🚀 Job ETL (Silver → Gold) finalizado com sucesso!
